# 2048 - Ejecución y pruebas (MEC)
Notebook para correr agentes y benchmarks con Poetry. Usa el mismo preset de pesos que `Main.py` (Expectimax d3, smooth_heavy).

## Parte 2: funciones de evaluación (MEC)
Las heurísticas están en `heuristics.py` y combinan varias señales para que Expectimax/Minimax prefieran tableros jugables. Se evalúan en log2 para que las potencias de 2 sean lineales. Componentes usados en `evaluate`:
- Vacías (`count_empty`): libera espacio para no quedar bloqueado.
- Monotonicidad (`monotonicity`): filas/columnas con orden creciente o decreciente consistente.
- Smoothness (`smoothness`): penaliza saltos grandes entre adyacentes (valor negativo).
- Esquina (`max_tile_in_corner`): recompensa si la ficha máxima queda en una esquina.
- Merges posibles (`merges_possible`): pares adyacentes listos para combinar.
- Peso posicional serpiente (`positional_weight`): layout deseado con fichas grandes agrupadas en una esquina.

Los pesos por defecto (`DEFAULT_WEIGHTS`) priorizan vacías + merges + esquina; `bench.py` permite sobreescribirlos por CLI o usar presets.

### Presets de pesos (bench.py)
Disponibles en `bench.py` para combinar heurísticas sin tocar código:
- `baseline`: similar a los defaults originales.
- `tuned`: refuerza merges y corner.
- `corner_heavy`: maximiza peso en la esquina.
- `smooth_heavy`: más peso a smoothness/monotonicidad (default que usamos).
- `snake`: activa peso posicional serpenteado.
Podés pasar `--preset` o `--weights empty=300,monotonicity=2,...` para probar combinaciones.
En la entrega dejamos `smooth_heavy` como preset base.


### Cómo documentamos las heurísticas
Para cumplir la parte de funciones de evaluación mostramos: componentes usadas, presets probados, y cómo varía el score al cambiar pesos. Abajo hay un tablero fijo para comparar presets y más abajo comandos listos para correr variantes (`baseline`, `snake`).


In [ ]:
# Ejemplo de sensibilidad de heurísticas en un tablero fijo
import numpy as np
from heuristics import (
    evaluate,
    count_empty, monotonicity, smoothness, max_tile_in_corner, merges_possible, positional_weight,
)

sample_grid = np.array([
    [2, 4, 8, 16],
    [32, 64, 128, 256],
    [2, 0, 0, 0],
    [0, 0, 0, 0],
], dtype=float)

weights_sets = {
    "baseline": {"empty": 250, "monotonicity": 1.5, "corner": 25, "smoothness": 3.0, "merges": 50},
    "smooth_heavy": {"empty": 280, "monotonicity": 2.2, "corner": 25, "smoothness": 4.0, "merges": 50},
    "snake": {"empty": 320, "monotonicity": 2.0, "corner": 40, "smoothness": 2.5, "merges": 70, "positional": 0.5},
}

components = {
    "empty": count_empty(sample_grid),
    "monotonicity": monotonicity(sample_grid),
    "smoothness": smoothness(sample_grid),
    "corner": max_tile_in_corner(sample_grid),
    "merges": merges_possible(sample_grid),
    "positional": positional_weight(sample_grid),
}
print("Componentes en sample_grid:")
for k, v in components.items():
    print(f"  {k}: {v}")

print("
Scores por preset:")
for name, w in weights_sets.items():
    print(f"  {name}: {evaluate(sample_grid, w):.2f}")


### Comandos para probar otros presets (opcional)
Usan 3 episodios para que el tiempo sea razonable; podés bajar a 1–2 si la máquina es lenta.

```bash
poetry run python bench.py --agent expectimax --depth 3 --episodes 3 --preset baseline --output expectimax_baseline_d3.csv
poetry run python bench.py --agent expectimax --depth 3 --episodes 3 --preset snake --output expectimax_snake_d3.csv
poetry run python summarize_results.py --pattern "expectimax_*d3.csv"
```
Luego, actualizar la tabla de resultados en esta sección (o agregar las filas nuevas).


## Parte 3: experimentación y registro de resultados
Metodología de pruebas para MEC (2048):
- Agentes: Expectimax depth=3 (default), Minimax depth=4 con/sin poda α-β.
- Pesos: preset `smooth_heavy` como base; otros presets para sensibilidad.
- Métricas por episodio: `win`, `max_tile`, `moves`, `duration_sec`, `grid_sum` (guardadas en CSV).
- Repeticiones: usamos 3 episodios para corridas rápidas; subir a 5–10 si hace falta más estabilidad para el informe.
- Nombres de archivos: `expectimax_smooth_heavy_d3.csv`, `minimax_smooth_heavy_d4_prune.csv`, `minimax_smooth_heavy_d4_noprune.csv`.
- Herramientas: `bench.py` genera CSV y `summarize_results.py` consolida en una tabla.


### Corridas realizadas (registro)
Resultados actuales (3 episodios, preset `smooth_heavy`):
- Expectimax d3: win 33.3%, max tile prom. 938.7, moves prom. 630.7, tiempo prom. 66.6s (`expectimax_smooth_heavy_d3.csv`).
- Minimax d4 con poda: win 0%, max tile prom. 597.3, moves prom. 577.3, tiempo prom. 225.6s (`minimax_smooth_heavy_d4_prune.csv`).
- Minimax d4 sin poda: win 0%, max tile prom. 682.7, moves prom. 550.3, tiempo prom. 684.2s (`minimax_smooth_heavy_d4_noprune.csv`).
Conclusión operativa: mantenemos Expectimax d3 + `smooth_heavy` como configuración por defecto; la poda acelera Minimax sin degradar la calidad observada.


In [ ]:
from datetime import datetime
from GameBoard import GameBoard
from Agent import Agent
from Expectimax_Agent import ExpectimaxAgent
from Minimax_AlphaBeta_Agent import MinimaxAlphaBetaAgent
from heuristics import evaluate

SMOOTH_HEAVY = {
    "empty": 280,
    "monotonicity": 2.2,
    "corner": 25,
    "smoothness": 4.0,
    "merges": 50,
}

int_to_string = ['UP', 'DOWN', 'LEFT', 'RIGHT']

def check_win(board: GameBoard):
    return board.get_max_tile() >= 2048

def play_once(agent: Agent, render: bool = True):
    board = GameBoard()
    done = False
    moves = 0
    if render:
        board.render()
    start = datetime.now()
    while not done:
        action = agent.play(board)
        if render:
            print(f"Next Action: {int_to_string[action]}  (move {moves})")
        done = board.play(action)
        done = done or check_win(board)
        if render:
            board.render()
        moves += 1
    duration = datetime.now() - start
    return {
        "moves": moves,
        "max_tile": board.get_max_tile(),
        "win": check_win(board),
        "duration": duration,
    }


## Partida rápida con Expectimax (depth=3, smooth_heavy)
Desactiva `render` si querés que corra sin imprimir el tablero.

In [ ]:
agent = ExpectimaxAgent(depth=3, weights=SMOOTH_HEAVY)
result = play_once(agent, render=False)
print(result)


## Benchmark Expectimax (poetry)
Ejemplo de corrida corta para generar CSV. Ajustar episodios/profundidad según necesidad.

In [ ]:
!poetry run python bench.py --agent expectimax --depth 3 --episodes 3 --preset smooth_heavy --output exp_expectimax_d3.csv


## Benchmark Minimax con poda α-β vs sin poda
Comparación para documentar el impacto de la poda. Misma profundidad/pesos.

In [ ]:
!poetry run python bench.py --agent minimax --depth 4 --episodes 3 --preset smooth_heavy --output exp_minimax_prune.csv


In [ ]:
!poetry run python bench.py --agent minimax --depth 4 --episodes 3 --preset smooth_heavy --no-pruning --output exp_minimax_noprune.csv


## Consolidar resultados
Usar el script de resumen para obtener win%, max tile promedio y tiempo promedio.

In [ ]:
!poetry run python summarize_results.py --pattern "exp_*.csv"
